<!-- PART 1 – OWNER: Asangika -->
# A. Title — TODO (Asangika)

Replace this cell with the title cell: course (PROG8431 – Data Analysis Mathematics, Algorithms and Modeling),
**Problem Analysis Workshop C – Transformation / Linear Regression**, Group 5, and the team table:

| Team member | Student ID |
|:--|--:|
| Asangika Hettiarachchi | 7461502 |
| Juan Antonio Sainz Galvez | 9072844 |
| Ved Vishal Patel | 9111628 |

Add the research question: *Which property and community characteristics explain residential listing prices in Ontario?*

# B. Setup and configuration — TODO (Asangika)

One code cell with **all** imports and settings used anywhere in the notebook. Keep at least the imports below
(Part 0 and Part 2 depend on them), then add a short markdown cell explaining the setup.

In [ ]:
# TODO (Asangika): complete this setup cell. These imports are required by Part 0 and Part 2.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import boxcox, ks_2samp, chi2_contingency, shapiro
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import BallTree
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 60)
RANDOM_STATE = 6174          # same seed used in the class demonstration
DATA_DIR = Path("data")      # relative path: never use C:\...

<!-- PART 0 – OWNER: Antonio -->
# C. Data sources, cleaning and integration (Antonio)

Our term-project dataset only has five fields per listing (address, area name, price, latitude, longitude).
To reach **at least 25 usable variables** we combine it with two public sources and derive location features.

| # | Source | What it adds | File |
|:-:|:--|:--|:--|
| 1 | Kaggle – *Ontario Residential Property Data* (mnabaee/ontarioproperties) | Listing price, address, area name, latitude and longitude (25,351 listings) | `data/properties.csv` |
| 2 | Statistics Canada – *Census Profile 2021*, census subdivisions (municipalities), Census Profile SDMX API (`api.statcan.gc.ca`) | 31 municipal indicators: population, density, age, dwelling types, incomes, low income, owners/renters, shelter costs, dwelling value, rent, immigration, education, labour force, commuting | `data/census_2021_csd_ontario.csv` |
| 3 | Wikidata – property P571 (*inception*) for each Ontario census subdivision, plus six manually added municipal incorporation dates (marked in `date_source`) | A real calendar date per municipality (used for the Julian date section) | `data/municipality_inception_dates.csv` |
| 4 | Derived from coordinates and address | Distance to downtown Toronto and to downtown Kitchener (haversine, km), condo/unit flag, region, city size class | computed below |

**Rationale.** A listing price depends on the property *and* on the community around it. Census indicators describe that
community (income, density, housing stock, commuting), and the distances capture the pull of the Toronto and
Kitchener-Waterloo job markets – the comparison our term project is about.

**How the join works.** Each address ends with its municipality (e.g. `"... Waterloo, ON"`). We match the end of the
address to a Statistics Canada census-subdivision name (longest match wins). About 11% of addresses use a former
municipality or community name (e.g. Kanata, Orleans, Ancaster); those listings take the municipality of their
nearest matched neighbour by latitude/longitude. `scripts/fetch_external_data.py` shows how sources 2 and 3 were downloaded.

In [ ]:
TORONTO_DOWNTOWN = (43.6532, -79.3832)
KITCHENER_DOWNTOWN = (43.4516, -80.4925)
ONTARIO_LAT_RANGE = (41.6, 56.9)
ONTARIO_LNG_RANGE = (-95.2, -74.3)
MIN_VALID_PRICE = 10_000


def load_properties(path):
    """Read the Kaggle listings file and give the columns clear snake_case names."""
    listings = pd.read_csv(path, index_col=0)
    return listings.rename(columns={"Address": "address", "AreaName": "area_name",
                                    "Price ($)": "price", "lat": "lat", "lng": "lng"})


def clean_properties(listings):
    """Apply the Group 5 cleaning rules and report how many rows each rule removes."""
    rules = {
        "missing coordinates (-999)": (listings["lat"] != -999) & (listings["lng"] != -999),
        "coordinates outside Ontario": listings["lat"].between(*ONTARIO_LAT_RANGE)
                                       & listings["lng"].between(*ONTARIO_LNG_RANGE),
        f"price under ${MIN_VALID_PRICE:,}": listings["price"] >= MIN_VALID_PRICE,
    }
    keep = pd.Series(True, index=listings.index)
    for rule_name, rule_mask in rules.items():
        removed = (keep & ~rule_mask).sum()
        print(f"Removed {removed:>5,} rows: {rule_name}")
        keep &= rule_mask
    cleaned = listings[keep].copy()
    print(f"Rows kept: {len(cleaned):,} of {len(listings):,}")
    return cleaned


def haversine_km(lat, lng, point):
    """Great-circle distance in km from each (lat, lng) to a fixed (lat, lng) point."""
    lat1, lng1 = np.radians(lat), np.radians(lng)
    lat2, lng2 = np.radians(point[0]), np.radians(point[1])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def build_name_lookup(census):
    """Map lower-case municipality names (and their English part before ' / ') to census codes."""
    lookup = {}
    for code_value, name in zip(census["csd_code"], census["csd_name"]):
        for variant in {name, name.split(" / ")[0]}:
            lookup.setdefault(variant.lower(), []).append(code_value)
    return lookup


def match_municipality(address, lookup, population):
    """Return the census code whose name matches the end of the address (longest match, largest population)."""
    words = address.replace(", ON", "").replace(",ON", "").split()
    for size in range(min(5, len(words)), 0, -1):
        candidate = " ".join(words[-size:]).lower()
        if candidate in lookup:
            return max(lookup[candidate], key=lambda c: population.get(c, 0))
    return np.nan


def fill_by_nearest_neighbour(listings, column):
    """Give unmatched listings the value of their nearest matched listing (haversine on lat/lng)."""
    matched = listings[listings[column].notna()]
    unmatched = listings[listings[column].isna()]
    tree = BallTree(np.radians(matched[["lat", "lng"]].values), metric="haversine")
    _, nearest = tree.query(np.radians(unmatched[["lat", "lng"]].values), k=1)
    listings.loc[unmatched.index, column] = matched[column].values[nearest[:, 0]]
    return listings


def assign_region(division_code):
    """Group Statistics Canada census divisions into eight Ontario regions (snake_case labels for clean dummy names)."""
    if division_code == 3520:
        return "toronto"
    if division_code in (3518, 3519, 3521, 3524):
        return "gta_excl_toronto"
    if division_code == 3530:
        return "waterloo_region"
    if division_code in (3522, 3523, 3525, 3526, 3528, 3529):
        return "golden_horseshoe_other"
    if 3531 <= division_code <= 3542:
        return "southwestern"
    if division_code <= 3514 or division_code == 3547:
        return "eastern"
    if division_code in (3515, 3516, 3543, 3544, 3546):
        return "central"
    return "northern"


def size_class(population):
    """Ordered size band of the municipality, based on its 2021 population."""
    if population < 10_000:
        return "Small (<10k)"
    if population < 100_000:
        return "Medium (10k-100k)"
    if population < 500_000:
        return "Large (100k-500k)"
    return "Very large (500k+)"

In [ ]:
# 1) Read the three sources
listings_raw = load_properties(DATA_DIR / "properties.csv")
census = pd.read_csv(DATA_DIR / "census_2021_csd_ontario.csv")
inception_dates = pd.read_csv(DATA_DIR / "municipality_inception_dates.csv")
print(f"Listings: {listings_raw.shape} | Census municipalities: {census.shape} | Inception dates: {inception_dates.shape}")

# 2) Clean the listings (same rules as Workshop B)
listings = clean_properties(listings_raw)

# 3) Match every listing to a census subdivision (municipality)
population_by_code = dict(zip(census["csd_code"], census["population_2021"]))
name_lookup = build_name_lookup(census)
listings["csd_code"] = listings["address"].apply(match_municipality, args=(name_lookup, population_by_code))
print(f"Matched by address: {listings['csd_code'].notna().mean():.1%}")
listings = fill_by_nearest_neighbour(listings, "csd_code")
listings["csd_code"] = listings["csd_code"].astype(int)

In [ ]:
# 4) Fill the few missing census values (suppressed by Statistics Canada for very small places) with the column median
census_columns = [c for c in census.columns if c not in ("csd_code", "csd_name")]
missing_before = census[census_columns].isna().sum().sum()
census[census_columns] = census[census_columns].fillna(census[census_columns].median())
print(f"Census cells filled with the column median: {missing_before}")

# 5) Join census indicators and inception dates; derive location features
study_df = (listings
            .merge(census, on="csd_code", how="left")
            .merge(inception_dates[["csd_code", "municipality_inception_date"]], on="csd_code", how="left")
            .rename(columns={"csd_name": "municipality"}))

study_df["is_condo_unit"] = study_df["address"].str.strip().str.startswith("#")
study_df["distance_to_toronto_km"] = haversine_km(study_df["lat"], study_df["lng"], TORONTO_DOWNTOWN)
study_df["distance_to_kitchener_km"] = haversine_km(study_df["lat"], study_df["lng"], KITCHENER_DOWNTOWN)
study_df["region"] = (study_df["csd_code"] // 1000).apply(assign_region)
study_df["city_size_class"] = study_df["population_2021"].apply(size_class)

id_columns = ["address", "area_name", "csd_code", "municipality"]
variable_columns = [c for c in study_df.columns if c not in id_columns]
study_df = study_df[id_columns + variable_columns]
study_df.to_csv(DATA_DIR / "study_dataset.csv", index=False)

print(f"Study file: {study_df.shape[0]:,} listings x {len(variable_columns)} variables "
      f"(+ {len(id_columns)} identifier columns) -> data/study_dataset.csv")
print(f"Listings without an inception date: {study_df['municipality_inception_date'].isna().sum():,}")

In [ ]:
# Data dictionary of the study file
variable_types = pd.DataFrame({
    "dtype": study_df[variable_columns].dtypes.astype(str),
    "missing": study_df[variable_columns].isna().sum(),
    "example": study_df[variable_columns].iloc[0],
})
variable_types["kind"] = np.select(
    [variable_types.index == "municipality_inception_date",
     variable_types["dtype"].isin(["object", "str", "bool"])],
    ["date", "categorical / flag"], default="numeric")
variable_types

**What Part 0 delivers.** `study_df` (also saved as `data/study_dataset.csv`) has one row per cleaned listing,
**40 variables** plus 4 identifier columns: the price (our dependent variable), coordinates, 2 distances, a condo flag,
2 categorical variables (`region`, `city_size_class`), one calendar date (`municipality_inception_date`) and 31 census
indicators. Cleaning removed rows with missing (-999) or out-of-Ontario coordinates and prices under $10,000.
A few census values suppressed for very small municipalities were filled with the column median. Some municipalities have
no inception date in Wikidata; Section F handles those.

<!-- PART 1 – OWNER: Asangika -->
# D. Data exploration: distributions — TODO (Asangika)

* `study_df.describe()` and `study_df.info()`
* Histograms **and** box plots of the main numeric variables (at least: `price`, `distance_to_toronto_km`,
  `population_density_per_km2`, `median_household_income`, `median_dwelling_value`, `pct_apartment_5plus_storeys`).
  Readable: one variable per subplot, titles, axis labels, prices shown in $ millions.
* Markdown (≈50–70 words): what the distributions show (skew, outliers) and why this motivates the transformations in H and I.

# E. Factor variables to numeric — TODO (Asangika)

* Convert the **ordered** factor `city_size_class` to a numeric code **`city_size_code`** (0 = Small … 3 = Very large)
  with `pd.Categorical(..., categories=[...], ordered=True).codes` (or a mapping dict).
* Convert the flag `is_condo_unit` (True/False) to 0/1 integers.
* Show before/after (e.g. `value_counts()` of both columns).
* Markdown: **50-word summary** – what was converted and why order matters.

# F. Calendar dates to Julian — TODO (Asangika)

* `pd.to_datetime(study_df["municipality_inception_date"])`, then `.to_julian_date()` (as in class) into a new column
  **`municipality_inception_julian`**.
* Some municipalities have no date: fill the missing Julian values with the **median** Julian date and add a 0/1 flag
  **`inception_date_missing`**.
* Show a few rows before/after.
* Markdown: **50-word summary** – what a Julian date is and why a model needs a number instead of a date.

# G. Categorical values to dummies — TODO (Asangika)

* `pd.get_dummies(study_df["region"], prefix="region", drop_first=True, dtype=int)`.
* Build the modelling table **`model_df`**: all numeric columns + the new codes/flags/dummies. Drop the text and
  identifier columns (`address`, `area_name`, `csd_code`, `municipality`, `region`, `city_size_class`,
  `municipality_inception_date`). Every column of `model_df` must be numeric. Keep `price` as it is (Part 2 transforms it).
* Markdown: **50-word summary** – what dummies are and why `drop_first=True` avoids the dummy-variable trap.

**Contract with Part 2 – the check cell below must pass before you push.**

In [ ]:
# Contract check between Part 1 and Part 2 (do not edit)
required_columns = {"price", "city_size_code", "is_condo_unit", "municipality_inception_julian",
                    "inception_date_missing", "distance_to_toronto_km", "median_dwelling_value"}
assert required_columns <= set(model_df.columns), f"Missing: {required_columns - set(model_df.columns)}"
assert any(c.startswith("region_") for c in model_df.columns), "Region dummies are missing"
assert all(pd.api.types.is_numeric_dtype(t) for t in model_df.dtypes), "model_df must be fully numeric"
assert model_df.isna().sum().sum() == 0, "model_df must not contain missing values"
print(f"model_df ready: {model_df.shape[0]:,} rows x {model_df.shape[1]} numeric columns")

<!-- PART 2 – OWNER: Antonio -->
# H. Box-Cox transformation (Antonio)

Prices and densities are strongly right-skewed (Section D). Linear regression works best when the dependent variable
and its residuals are roughly symmetric. The Box-Cox transformation $y^{(\lambda)} = \frac{y^\lambda - 1}{\lambda}$
($\ln y$ when $\lambda = 0$) searches for the power $\lambda$ that makes a **positive** variable closest to normal.

In [ ]:
def skew_report(series_by_name):
    """Skewness and Shapiro-Wilk W of each series (Shapiro on a 5,000-row sample, SciPy's recommended maximum)."""
    rows = {}
    for name, values in series_by_name.items():
        sample = values.sample(min(5_000, len(values)), random_state=RANDOM_STATE)
        w_stat, p_value = shapiro(sample)
        rows[name] = {"skewness": stats.skew(values), "shapiro_W": w_stat, "shapiro_p": p_value}
    return pd.DataFrame(rows).T


def plot_before_after(before, after, title_before, title_after, xlabel_before, xlabel_after):
    """Two histograms side by side: original variable and transformed variable."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(before, bins=60, color="#4C72B0")
    axes[0].set(title=title_before, xlabel=xlabel_before, ylabel="Listings")
    axes[1].hist(after, bins=60, color="#55A868")
    axes[1].set(title=title_after, xlabel=xlabel_after, ylabel="Listings")
    plt.tight_layout()
    plt.show()


boxcox_columns = ["price", "population_density_per_km2", "distance_to_toronto_km"]
boxcox_lambdas = {}
for column in boxcox_columns:
    transformed, fitted_lambda = boxcox(model_df[column])   # all three columns are strictly positive
    model_df[f"{column}_boxcox"] = transformed
    boxcox_lambdas[column] = fitted_lambda

display(pd.Series(boxcox_lambdas, name="Box-Cox lambda").to_frame())
display(skew_report({**{c: model_df[c] for c in boxcox_columns},
                     **{f"{c}_boxcox": model_df[f"{c}_boxcox"] for c in boxcox_columns}}))

plot_before_after(model_df["price"] / 1e6, model_df["price_boxcox"],
                  "Price – original", f"Price – Box-Cox (lambda = {boxcox_lambdas['price']:.3f})",
                  "Price ($ millions)", "Box-Cox price")

**Box-Cox summary (≈50 words).** Box-Cox chose λ = 0.022 for price (almost a log), 0.290 for population density and 0.274 for distance to Toronto.
Price skewness fell from 9.85 to 0.01 and the Shapiro-Wilk W rose from 0.50 to 0.98, so the transformed price is close to
symmetric. Shapiro-Wilk still rejects exact normality – expected with thousands of listings – but the long right tail is gone.

# I. Tukey's Ladder of Powers (Antonio)

Tukey's ladder tries a fixed set of powers – $x^2,\ x,\ \sqrt{x},\ \ln x,\ 1/\sqrt{x},\ 1/x$ – and keeps the rung
whose result is closest to symmetric (skewness nearest 0). Unlike Box-Cox it gives an easy-to-explain transformation.
We apply it to `population_2021` and `pct_apartment_5plus_storeys` (strongly right-skewed census indicators).

In [ ]:
TUKEY_LADDER = {
    "x^2": lambda x: x ** 2,
    "x (no change)": lambda x: x,
    "sqrt(x)": np.sqrt,
    "log(x)": np.log,
    "-1/sqrt(x)": lambda x: -1 / np.sqrt(x),
    "-1/x": lambda x: -1 / x,
}


def tukey_ladder(values):
    """Skewness of the variable under each rung of Tukey's ladder (values are shifted by +1 if any are 0)."""
    shifted = values + 1 if (values <= 0).any() else values
    return pd.Series({rung: stats.skew(function(shifted)) for rung, function in TUKEY_LADDER.items()},
                     name="skewness")


tukey_columns = ["population_2021", "pct_apartment_5plus_storeys"]
ladder_table = pd.concat({column: tukey_ladder(model_df[column]) for column in tukey_columns}, axis=1)
display(ladder_table)

tukey_choice = ladder_table.abs().idxmin()
print("Rung with skewness closest to 0:")
print(tukey_choice.to_string())

for column, rung in tukey_choice.items():
    values = model_df[column] + 1 if (model_df[column] <= 0).any() else model_df[column]
    model_df[f"{column}_tukey"] = TUKEY_LADDER[rung](values)

plot_before_after(model_df["pct_apartment_5plus_storeys"], model_df["pct_apartment_5plus_storeys_tukey"],
                  "Apartments in 5+ storey buildings – original",
                  f"After Tukey rung {tukey_choice['pct_apartment_5plus_storeys']}",
                  "% of dwellings", "Transformed value")

**Tukey's Ladder summary (≈50 words).** For `population_2021` the rung with skewness closest to zero is **log(x)** (skewness 1.17 → −0.57); for
`pct_apartment_5plus_storeys` it is **√x** (0.69 → −0.001). Both new `_tukey` columns replace the originals in the MLR.
Tukey's ladder follows the same idea as Box-Cox, but with simple, explainable powers instead of an estimated λ.

# J. Train / test split and comparability (Antonio)

As in class, we split the data **80% train / 20% test** with a fixed `random_state` (so everyone gets the same split).
The model is built on the training set only; the test set measures how well it predicts listings it has never seen.
We then check that both sets are comparable: summaries, **Kolmogorov–Smirnov** for numeric variables
(H₀: same distribution) and **chi-square** for categorical variables (H₀: same proportions).

In [ ]:
train_df, test_df = train_test_split(model_df, test_size=0.20, random_state=RANDOM_STATE)
print(f"Train: {train_df.shape} | Test: {test_df.shape}")

key_numeric = ["price", "price_boxcox", "distance_to_toronto_km", "population_density_per_km2",
               "median_household_income", "median_dwelling_value", "municipality_inception_julian"]
summary_comparison = pd.concat({"train mean": train_df[key_numeric].mean(), "test mean": test_df[key_numeric].mean(),
                                "train median": train_df[key_numeric].median(), "test median": test_df[key_numeric].median()},
                               axis=1)
display(summary_comparison)


def ks_comparison(train, test, columns):
    """Kolmogorov-Smirnov test for every numeric column; H0 = both samples come from the same distribution."""
    results = {c: ks_2samp(train[c], test[c]) for c in columns}
    table = pd.DataFrame({"KS statistic": {c: r.statistic for c, r in results.items()},
                          "p-value": {c: r.pvalue for c, r in results.items()}})
    table["different at 5%?"] = np.where(table["p-value"] < 0.05, "yes", "no")
    return table


binary_columns = [c for c in model_df.columns if c.startswith("region_")] + ["is_condo_unit", "inception_date_missing"]
continuous_columns = [c for c in model_df.columns if c not in binary_columns + ["city_size_code"]]
ks_table = ks_comparison(train_df, test_df, continuous_columns)
print(f"Numeric variables with a significant train/test difference: {(ks_table['p-value'] < 0.05).sum()} of {len(ks_table)}")
display(ks_table.sort_values("p-value").head(10))


def chi_square_comparison(train, test, columns):
    """Chi-square test of train vs test counts for each categorical/binary column; H0 = same proportions."""
    rows = {}
    for column in columns:
        table = pd.DataFrame({"train": train[column].value_counts(), "test": test[column].value_counts()}).fillna(0)
        chi2, p_value, dof, _ = chi2_contingency(table.T)
        rows[column] = {"chi-square": chi2, "dof": dof, "p-value": p_value}
    out = pd.DataFrame(rows).T
    out["different at 5%?"] = np.where(out["p-value"] < 0.05, "yes", "no")
    return out


region_counts = pd.DataFrame({"train": train_df[[c for c in binary_columns if c.startswith("region_")]].sum(),
                              "test": test_df[[c for c in binary_columns if c.startswith("region_")]].sum()})
display(chi_square_comparison(train_df, test_df, binary_columns + ["city_size_code"]))

**Split discussion.** The 80/20 split gives 19,403 training and 4,851 test listings. Means and medians are almost identical
(median price $389,700 train vs $379,900 test). Kolmogorov–Smirnov finds no significant difference for 41 of 42 numeric
variables; the only exception, `land_area_km2` (p = 0.047), is borderline, so we note it and keep the split. Chi-square finds
no difference in regions, condo share, missing-date flag or city size. The two sets are comparable, so the test set is a fair
check of the models.

# K. Dependent variable (Antonio)

We predict the **listing price**. Because prices are strongly right-skewed, the models use the **Box-Cox transformed
price (`price_boxcox`)** from Section H as the dependent variable. RMSE is reported on this transformed scale (the scale
the model is fitted on) and, to make it understandable, also back-transformed to dollars.

In [ ]:
DEPENDENT = "price_boxcox"
PRICE_LAMBDA = boxcox_lambdas["price"]


def inverse_boxcox(values, fitted_lambda):
    """Convert Box-Cox values back to the original price scale ($)."""
    if abs(fitted_lambda) < 1e-12:
        return np.exp(values)
    return np.power(values * fitted_lambda + 1, 1 / fitted_lambda)


def evaluate_model(fitted_model, train, test, name):
    """Standard class measures for a fitted statsmodels OLS model, on train and test."""
    pred_train = fitted_model.predict(train)
    pred_test = fitted_model.predict(test)
    return pd.Series({
        "predictors": int(fitted_model.df_model),
        "F-statistic": fitted_model.fvalue,
        "Prob (F-statistic)": fitted_model.f_pvalue,
        "R-squared": fitted_model.rsquared,
        "Adj. R-squared": fitted_model.rsquared_adj,
        "AIC": fitted_model.aic,
        "RMSE train (Box-Cox)": np.sqrt(mean_squared_error(train[DEPENDENT], pred_train)),
        "RMSE test (Box-Cox)": np.sqrt(mean_squared_error(test[DEPENDENT], pred_test)),
        "Median abs. error test ($)": np.median(np.abs(test["price"] - inverse_boxcox(pred_test, PRICE_LAMBDA))),
    }, name=name)


# Correlation of every candidate predictor with the dependent variable (training set only)
candidate_predictors = [c for c in model_df.columns if c not in ("price", "price_boxcox")]
correlations = train_df[candidate_predictors].corrwith(train_df[DEPENDENT]).sort_values(key=np.abs, ascending=False)
correlations.head(12).to_frame("correlation with price_boxcox")

# L. Two simple linear regression (SLR) models (Antonio)

We build one SLR for each of two **different** predictors, chosen from the correlation table above:
* **SLR 1 – `median_dwelling_value`** (census median value of dwellings in the municipality): the strongest correlation
  with the dependent variable (r = 0.43).
* **SLR 2 – `distance_to_toronto_km_boxcox`** (Box-Cox distance to downtown Toronto): a pure location measure (r = −0.33).

They capture different ideas: how valuable homes are in the community vs. how far it is from the main job market.

In [ ]:
SLR_PREDICTORS = ["median_dwelling_value", "distance_to_toronto_km_boxcox"]

slr_models = {}
for predictor in SLR_PREDICTORS:
    slr_models[predictor] = smf.ols(f"{DEPENDENT} ~ {predictor}", data=train_df).fit()
    print(slr_models[predictor].summary().tables[1])
    print()

slr_comparison = pd.concat([evaluate_model(model, train_df, test_df, f"SLR: {predictor}")
                            for predictor, model in slr_models.items()], axis=1)
slr_comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (predictor, model) in zip(axes, slr_models.items()):
    ax.scatter(train_df[predictor], train_df[DEPENDENT], s=4, alpha=0.25, label="training listings")
    x_line = np.linspace(train_df[predictor].min(), train_df[predictor].max(), 100)
    ax.plot(x_line, model.params["Intercept"] + model.params[predictor] * x_line, color="red", label="fitted line")
    ax.set(title=f"SLR: {DEPENDENT} ~ {predictor}  (R² = {model.rsquared:.3f})", xlabel=predictor, ylabel="Box-Cox price")
    ax.legend()
plt.tight_layout()
plt.show()

**SLR comparison – which model is superior and why.** Both models are significant (Prob F-statistic ≈ 0) and both slopes have the expected sign: prices are
higher where the community's median dwelling value is higher, and lower the farther the listing is from Toronto.
**SLR 1 (median dwelling value) is superior on every measure:** higher F-statistic (4,392 vs 2,359), higher R² and adjusted R²
(0.185 vs 0.108), lower AIC, and lower RMSE on both train (1.070 vs 1.119) and test (1.066 vs 1.112). In dollars, its median
absolute error on the test set is about $155k vs $171k. Train and test RMSE are almost equal in both models, so neither
overfits. Dwelling value wins because it already summarizes many local factors (location, incomes, housing type), while
distance alone misses expensive places far from Toronto (e.g. Muskoka, The Blue Mountains). Still, one variable explains
less than a fifth of the variation in price, which motivates the multiple regression.

# M. Multiple linear regression (MLR) and variable selection (Antonio)

**Full model.** As in class, we start with **all** independent variables. Original columns that were transformed in H and I
are replaced by their transformed versions, and `pct_renter_households` is left out because owners + renters ≈ 100%
(it would be an exact duplicate of `pct_owner_households`).

**Step 2 – Backward elimination (class technique).** Remove the predictor with the highest p-value, refit, and repeat
until every remaining predictor is significant at 5%.

**Step 3 – Coefficient check.** Following the class review of coefficients, we check that the signs make sense. Where several
predictors measure the same concept (three income measures, two age measures, population and dwellings, three
labour-force rates) they compete with each other and get opposite signs – a sign of multicollinearity (the large
condition number in the summary). We keep one measure per concept and run backward elimination again.

In [ ]:
replaced_by_transform = ["price", "price_boxcox", "population_density_per_km2", "distance_to_toronto_km",
                         "population_2021", "pct_apartment_5plus_storeys", "pct_renter_households"]
all_predictors = [c for c in model_df.columns if c not in replaced_by_transform]
print(f"Full model starts with {len(all_predictors)} predictors")

full_model = smf.ols(f"{DEPENDENT} ~ " + " + ".join(all_predictors), data=train_df).fit()
print(full_model.summary())

In [ ]:
def backward_elimination(train, predictors, alpha=0.05):
    """Drop the least significant predictor (highest p-value) one at a time until all p-values < alpha."""
    remaining = list(predictors)
    removal_log = []
    while True:
        model = smf.ols(f"{DEPENDENT} ~ " + " + ".join(remaining), data=train).fit()
        p_values = model.pvalues.drop("Intercept")
        worst = p_values.idxmax()
        if p_values[worst] < alpha:
            return model, pd.DataFrame(removal_log)
        removal_log.append({"step": len(removal_log) + 1, "removed": worst, "p-value": p_values[worst],
                            "Adj. R² before removal": model.rsquared_adj})
        remaining.remove(worst)


backward_model, removal_log = backward_elimination(train_df, all_predictors)
display(removal_log)
print(backward_model.summary())

In [ ]:
# Step 3: one measure per concept, then backward elimination again
DUPLICATE_CONCEPT_MEASURES = pd.DataFrame([
    ("median_household_income", "income – income is already captured by dwelling value, rent, shelter cost and low-income share"),
    ("average_household_income", "income – duplicate of the median income measures"),
    ("median_after_tax_household_income", "income – duplicate of the other income measures"),
    ("median_age", "age – duplicate of average_age"),
    ("private_dwellings", "size – duplicate of population (and of city_size_code)"),
    ("unemployment_rate", "labour force – equals 100 minus employment / participation ratio"),
], columns=["removed predictor", "reason"])
display(DUPLICATE_CONCEPT_MEASURES)

refined_predictors = [c for c in all_predictors if c not in set(DUPLICATE_CONCEPT_MEASURES["removed predictor"])]
final_model, refined_removal_log = backward_elimination(train_df, refined_predictors)
display(refined_removal_log)
print(final_model.summary())

best_slr_name = slr_comparison.loc["RMSE test (Box-Cox)"].idxmin()
model_comparison = pd.concat([
    slr_comparison[best_slr_name],
    evaluate_model(full_model, train_df, test_df, "Full MLR"),
    evaluate_model(backward_model, train_df, test_df, "MLR after backward elimination"),
    evaluate_model(final_model, train_df, test_df, "Final MLR (one measure per concept)"),
], axis=1)
display(model_comparison)

final_coefficients = pd.DataFrame({"coefficient": final_model.params, "p-value": final_model.pvalues}).drop("Intercept")
display(final_coefficients.sort_values("coefficient", key=np.abs, ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
test_predictions = final_model.predict(test_df)
axes[0].scatter(test_predictions, test_df[DEPENDENT], s=4, alpha=0.3)
limits = [test_df[DEPENDENT].min(), test_df[DEPENDENT].max()]
axes[0].plot(limits, limits, color="red")
axes[0].set(title="Final MLR – predicted vs actual (test set)", xlabel="Predicted Box-Cox price", ylabel="Actual Box-Cox price")
axes[1].hist(final_model.resid, bins=60)
axes[1].set(title="Final MLR – residuals (training set)", xlabel="Residual", ylabel="Listings")
plt.tight_layout()
plt.show()

**MLR discussion – evidence for the final model.** 
* **Full model (45 predictors):** significant overall (Prob F ≈ 0), adjusted R² 0.264, but most predictors have p > 0.05 and
  the condition number is very large – many predictors repeat the same information.
* **Backward elimination:** 25 predictors removed one at a time; 20 remain, all with p < 0.05. Adjusted R² stays 0.264 and
  test RMSE 1.001 – removing them cost nothing.
* **Coefficient check:** that model still contained competing measures of the same concept with opposite signs (after-tax
  income + vs average income −, average age + vs median age −). Keeping one measure per concept and eliminating again gives the
  **final model: 16 predictors, all p < 0.05, F = 434, adjusted R² 0.263, RMSE train 1.016 / test 1.002** – the same accuracy
  as the 45-predictor model with about a third of the variables.
* **Signs make sense:** condo units sell for much less (−0.91 on a near-log scale, roughly 60% lower), prices fall with
  distance to Toronto and with the share of low-income households, and rise with median dwelling value, rent, owner shelter
  costs and rooms per dwelling. The pair employment rate (−) / participation rate (+) is still correlated, so we interpret it
  as one labour-market effect.
* **Versus the best SLR:** adjusted R² 0.185 → 0.263, test RMSE 1.066 → 1.002, median absolute test error ≈ $155k → $133k.
  Train ≈ test RMSE, so the model does not overfit.
* **Limitation:** R² ≈ 0.26 means most of the price variation comes from the property itself (size, bedrooms, lot), which our
  dataset does not have; census values are municipal averages shared by many listings.

**Final choice: the 16-predictor model** – simplest model with every predictor significant, sensible signs and the same
test performance as the larger models.

<!-- PART 3 – OWNER: Ved -->
# N. Summary of transformations — TODO (Ved)

**50-word summary** of every transformation performed on the data, in order, and why: cleaning and integration (C),
factor to numeric (E), Julian dates (F), dummies (G), Box-Cox (H), Tukey's Ladder (I).

# O. Model comparison table — TODO (Ved)

One table like the professor's whiteboard. Rows: Prob (F-statistic), Adj. R², significant predictors / total,
coefficient signs that make sense (yes/no), RMSE train, RMSE test. Columns: SLR 1, SLR 2, Full MLR, Final MLR.
Use the values already computed in L and M (`slr_comparison`, `model_comparison`) – do not retype numbers by hand.

# P. Conclusions and limitations — TODO (Ved)

Which model we recommend and why; what it says about Ontario prices (Toronto and Kitchener-Waterloo vs the rest);
limitations (listing prices are asking prices, census data are municipal averages, one listing date); next steps for the term project.

# Q. README and R. presentation run sheet — TODO (Ved)

* `README.md` in this folder: goal, data sources with links, how to run, contributions table (same style as Workshop B).
* Last cell of the notebook: speaker per section and seconds per section, total **7:00**
  (Asangika A–G ≈ 2:00 · Antonio H–M ≈ 3:00 · Ved N–P ≈ 2:00).